In [14]:
from datasets import Dataset, DatasetDict
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset
import pickle 
import pandas as pd
import mlflow

In [15]:
mlflow.set_experiment("adeptID")

<Experiment: artifact_location='file:///c:/Users/jvhua/OneDrive/Desktop/ISYE-CSE-MGT-6748-Group-1/preprocess/mlruns/936985963919367918', creation_time=1718591276177, experiment_id='936985963919367918', last_update_time=1718591276177, lifecycle_stage='active', name='adeptID', tags={}>

In [4]:
df = pd.read_excel(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\set-fit-data.xlsx")

In [5]:
df = df[['uuid', 'string_chunk', 'label']].rename(columns = {'string_chunk': 'text'})

In [6]:
dataset = Dataset.from_pandas(df)

In [7]:
train_val_test_split = dataset.train_test_split(test_size=0.5)

train_val_split = train_val_test_split['train'].train_test_split(test_size=0.8)

In [8]:
dataset_dict = DatasetDict({
    'train': train_val_split['train'],
    'validation': train_val_split['test'],
    'test': train_val_test_split['test']
})

In [9]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['uuid', 'text', 'label'],
        num_rows: 30
    })
    validation: Dataset({
        features: ['uuid', 'text', 'label'],
        num_rows: 120
    })
    test: Dataset({
        features: ['uuid', 'text', 'label'],
        num_rows: 150
    })
})

In [10]:
# Simulate the few-shot regime by sampling 8 examples per class
train_dataset = sample_dataset(dataset_dict["train"], label_column="label", num_samples=8)
eval_dataset = dataset_dict["validation"].select(range(100))
test_dataset = dataset_dict["test"].select(range(100))

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\setfit\data.py:154: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.apply(lambda x: x.sample(min(num_samples, len(x)), random_state=seed))


In [11]:
# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "nomic-ai/nomic-embed-text-v1.5", 
    labels=[0, 1],
    trust_remote_code = True
)

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
<All keys matched successfully>
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [12]:
args = TrainingArguments(
    batch_size=8,
    num_epochs=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy",
    column_mapping={"text": "text", "label": "label"}
)

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

In [16]:
with mlflow.start_run(run_name = "setfit-preprocess-model-train"):
    trainer.train()
    metrics = trainer.evaluate(test_dataset)
    mlflow.log_metric('accuracy',  metrics['accuracy'])
    print(metrics)

***** Running training *****
  Num unique pairs = 144
  Batch size = 8
  Num epochs = 10
  Total optimization steps = 180


  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

{'embedding_loss': 0.1349, 'learning_rate': 1.111111111111111e-06, 'epoch': 0.06}


  0%|          | 0/639 [00:00<?, ?it/s]

...

{'eval_embedding_loss': 0.2368, 'learning_rate': 0.0, 'epoch': 10.0}


Loading best SentenceTransformer model from step 36.
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
<All keys matched successfully>


{'train_runtime': 1241.5012, 'train_samples_per_second': 1.16, 'train_steps_per_second': 0.145, 'epoch': 10.0}


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.7}


In [18]:
save_directory = r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\setfit_300_nomic_model"
trainer.model._save_pretrained(save_directory=save_directory)

In [23]:

model = SetFitModel.from_pretrained(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\setfit_300_nomic_model", trust_remote_code = True)

<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
<All keys matched successfully>


[1, 0]


In [24]:
preds = model.predict(["for this job you will need a AAA certification","this job has a salary of 100,000 a year", "we are looking for team leadership", "in this job it is a remote oppurtunity", 'you will be the best you available', 'we offer a salary of 20 dollars an hour', 'we do not consider age and gender or race we are an equal oppurtunity employer'])
print(preds)

[0, 1, 0, 0, 0, 1, 0]
